# Chapter 9 Decorators and Closures

## Topic 1: Decorators 101 (Basic Mechanics and Syntactic Sugar)
1. What It Is
- A decorator is a callable (a function or an object implementing `__call__`) that takes another **function** as an argument (the decorated function).
- A decorator performs some processing on or with the decorated function, and either returns the original function or replaces it with a completely different callable object.

2. The Underlying Mechanism: Syntactic Sugar and Rebinding
- In Python, the `@` symbol used for decorators is purely syntactic sugar. When CPython compiles and executes a module containing a decorated function:
```python
@decorate
def target():
    print('running target()')
```
It is evaluated as exact syntactic sugar for defining the original function and immediately passing it into the decorator function:

```python
def target():
    print('running target()')
```
`target = decorate(target)`

- Line-by-Line Execution Mechanics:
    1. Instantiation: Python creates the initial function object in memory (named target).
    Invocation: Python immediately passes that newly created function object as the first positional argument to decorate(target).
    2. Rebinding: Python takes whatever decorate(target) returns and rebinds the symbol target to that returned object.
    3. If decorate returns an inner wrapper function object inner, the variable name target no longer points to the original function—it now references inner.

3. Gotchas & Edge Cases
- Loss of Original Function Reference: Beginners often mistakenly view decorators as "attaching metadata" to a function while leaving the function intact. In reality, unless the decorator explicitly returns func unchanged (a registration decorator), the original function is replaced.
- Attribute Masking (`__name__` and `__doc__`): Because target is rebound to the replacement callable (e.g., `inner`), accessing `target.__name__` will evaluate to 'inner' rather than 'target', and docstrings on the original function will be lost unless copied.

In [12]:
def make_header(func):
    def inner():
        return "HEADER: " + func()
    return inner

def get_body():
    return "main body"

# Manual invocation without @ syntax
assigned_fn = make_header(get_body)

# Decorated invocation using @ syntax
@make_header
def get_footer():
    return "footer text"

# Print #1
print(1, get_footer()) # HEADER: footer text

# Print #2
print(2, get_footer.__name__, get_body.__name__) # inner get_body

# Print #3
print(3, get_body() == assigned_fn()) # False

1 HEADER: footer text
2 inner get_body
3 False


In [4]:
from typing import Callable

register: list[Callable] = []

def reg(inner: Callable):
    print(f'registering {inner}')
    register.append(inner)
    return inner

@reg
def f1():
    print('running f1')
@reg
def f2():
    print('running f2')



registering <function f1 at 0x10f9bbec0>
registering <function f2 at 0x10f9ba660>


In [5]:
def f(a):
    print(a)
    print(b)

f(1)

1


NameError: name 'b' is not defined

In [6]:
b = 3
f(2) # Now it works as b is now defined in the global scope and note that b is not defined in function scope of f.

2
3


In [11]:
def f2(a):
    print(a)
    print(b) #NOTE: Python assume a variable defined within a function body is local! 
    b = 9
b=100
f2(2)

2


UnboundLocalError: cannot access local variable 'b' where it is not associated with a value

In [10]:
def f3(a):
    print(a)
    global b #NOTE: to tell python to use global scope for searching b.
    print(b)
    b = 9

print(f3(2))
print(b)

2
9
None
9


## Topic 2: When Python Executes Decorators (Import Time vs. Runtime)

## 1. What It Is
A key feature of decorators is that they execute **immediately at import time** as soon as the module is loaded by Python, whereas decorated functions execute at **runtime** when explicitly invoked [passage 433, 435].

## 2. The Mechanism: Registration vs. Wrapper Decorators

### A. Registration Decorators (Import-Time Side Effects)
Registration decorators accept a function, add it to an internal registry (such as a list, dictionary, or route map), and return the function **unchanged** [passage 433, 436].
- **Import Time:** The decorator runs and registers the function [passage 433, 435].
- **Runtime:** Calling the function executes the original code directly with zero wrapper overhead [passage 434, 436].

### B. Wrapper Decorators (Runtime Interception)
Wrapper decorators create and return a new inner function object that replaces the target function [passage 431, 436].
- **Import Time:** The decorator executes, instantiates the inner wrapper, and rebinds the target symbol [passage 431, 432, 435].
- **Runtime:** Invoking the target symbol executes the inner wrapper logic [passage 431, 459].

## 3. Gotchas & Edge Cases
* **Import-Time Performance Degradation:** Placing heavy computational logic, network requests, or database queries directly inside the body of a decorator (outside the inner wrapper) forces those heavy operations to execute during **module import** [passage 433, 435]. This slows down application startup or crashes imports if external dependencies are missing [passage 435].

In [ ]:
### Challenge 2: Predict the Output

def log_import(func):
    print(f"IMPORT: decorating {func.__name__}")
    return func

def log_runtime(func):
    print(f"IMPORT: wrapping {func.__name__}")
    def wrapper(*args):
        print(f"RUNTIME: calling {func.__name__}")
        return func(*args)
    return wrapper

@log_import
def action_a():
    return "A"

@log_runtime
def action_b():
    return "B"

print("--- START MAIN ---")
res_a = action_a()
res_b = action_b()
print("--- END MAIN ---")


#1. List all lines printed in exact order from top to bottom.
# IMPORT: decorating action_a --- import time
# IMPORT: wrapping action_b ---- import time
# --- START MAIN --- runtime
# A #NOTE: this is wrong because res_a is not printed just returned!!!
# RUNTIME: calling action_b ----runtime
#---END MAIN --- runtime




#2. Which print lines occurred at import time vs. runtime?

## Topic 3: Variable Scope Rules (Local, Global, and UnboundLocalError)

1. ### What It Is
- Python determines variable scope **statically** at compile time [passage 440]. If a variable is assigned anywhere within a function body, Python treats it as a ***local variable*** throughout that entire function [passage 439, 440].

2. ### The Mechanism: Bytecode Compilation & Scope Determination
When CPython compiles a function body:
1. It scans for any assignment statements (`=`, `+=`, `-=`, etc.) [passage 439, 440, 451].
2. Any variable assigned to is flagged as a local variable, generating `LOAD_FAST` and `STORE_FAST` bytecode instructions [passage 440, 443, 444].
3. Unassigned variables referenced in the body are treated as global (`LOAD_GLOBAL`) or free variables (`LOAD_DEREF`) [passage 438, 442, 449].

If Python attempts to read a local variable before an assignment executes at runtime, it raises an `UnboundLocalError` [passage 439, 440].

### 3. Gotchas & Edge Cases
* **The Masking Trap:** Writing `b = b + 1` or executing `print(b)` followed by `b = 9` inside a function body causes `b` to be compiled as local throughout the function [passage 439, 440]. It will **not** fall back to reading a global variable `b` prior to the assignment line [passage 439, 440].
* **The `global` Declaration:** Using `global b` explicitly instructs the compiler to bind `b` to the module global scope [passage 441].


In [ ]:
### Challenge 3: Predict the Output

val = 100

def test_scope_a():
    print(1, val)

def test_scope_b():
    try:
        print(2, val)
        val = 200
    except UnboundLocalError:
        print(2, "UnboundLocalError")

def test_scope_c():
    global val
    val = 300

test_scope_a() # 100
test_scope_b() # UnboundedLocalError
test_scope_c() #
print(3, val) # 300




## Topic 4: Closures and Free Variables

### 1. What It Is
A closure is a function with an extended scope that retains variable bindings from an enclosing outer function body, even after that outer function has returned and its local scope is destroyed [passage 444, 448, 449].

### 2. The Mechanism: Free Variables and Cell Objects
- **Free Variable:** A variable referenced inside a nested function that is **neither** a local variable of that nested function **nor** a global variable [passage 444, 448].
- **Function Inspection:**
  - `fn.__code__.co_freevars`: Returns a tuple containing the names of free variables.
  - `fn.__code__.co_varnames`: Returns a tuple containing the names of local variables.
  - `fn.__closure__`: A tuple of `cell` objects corresponding to `co_freevars`.
  - `fn.__closure__[i].cell_contents`: Stores the actual live reference/value of the closed-over variable.

### 3. Gotchas & Edge Cases
* **Mutable vs. Immutable Free Variables:** Mutating a mutable object inside a closure (e.g., `series.append(x)`) works because the reference stored in `cell_contents` remains invariant [passage 447, 452]. However, attempting to rebind an immutable object (`count = count + 1`) compiles `count` as a local variable, causing an assignment error unless `nonlocal` is declared [passage 451, 452].

### 4. Currency Check (Grounded Only)
* **BOOK vs CURRENT (Python 3.13, PEP 667):**
  * **BOOK (p. 314):** Describes inspecting `__closure__` and `locals()` in closures [passage 449].
  * **CURRENT (Python 3.13 Documentation):** PEP 667 defined mutation semantics for `locals()` in optimized scopes [passage 187, 203]. Calling `locals()` inside a function/closure now explicitly returns an **independent snapshot**, and mutating that dictionary does not alter actual local or closure variables [passage 203, 204]. Accessing `FrameType.f_locals` returns a write-through proxy [passage 205].



In [ ]:
class Average:
    def __init__(self):
        self._items = []

    def __call__(self, item):
        self._items.append(item)
        return sum(self._items) / len(self._items)

avg = Average()
for i in range(1, 11):
    print(avg(i))

1.0
1.5
2.0
2.5
3.0
3.5
4.0
4.5
5.0
5.5


In [51]:
# functional implemention of average using higher order functions
def make_average():
    series = []
    def inner(value):
        series.append(value)
        return sum(series)/len(series)
    return inner

avg2 = make_average()
for i in range(1, 11):
    print(avg2(i))

avg2.__code__.co_freevars, avg2.__code__.co_varnames, avg2.__closure__[0].cell_contents

1.0
1.5
2.0
2.5
3.0
3.5
4.0
4.5
5.0
5.5


(('series',), ('value',), [1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

In [ ]:
def make_fib():
    seen = {}

    def inner(n):
        if n < 2:
            return n
        if n not in seen:
            seen[n] = inner(n-1) + inner(n-2)      
        return seen[n]
    
    return inner

fib = make_fib()
for i in range(11):
    print(fib(i))
print()

class FibNumber:
    def __init__(self):
        self.seen = {}
        
    def __call__(self, n: int) -> int:
        if n in self.seen:
            return self.seen[n]
        if n < 2:
            return n
        res = self.__call__(n-1) + self.__call__(n-2)
        self.seen[n] = res
        return res

fib_num = FibNumber()
for i in range(11):
    print(fib_num(i))


def fib3(n):
    if n < 2:
        return n
    
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a+b
    return b

print()
for i in range(11):
    fib3(i)



0
1
1
2
3
5
8
13
21
34
55

0
1
1
2
3
5
8
13
21
34
55



In [ ]:
### Challenge 4: Predict the Output

def make_accumulator():
    items = []
    def adder(val):
        items.append(val)
        return sum(items)
    return adder

acc1 = make_accumulator()
acc2 = make_accumulator()

acc1(10)
acc1(20)
acc2(100)

print(1, acc1.__code__.co_freevars) # (items,)
print(2, acc1.__closure__[0].cell_contents) # (10, 20)
print(3, acc2.__closure__[0].cell_contents) # (100,)


1 ('items',)
2 [10, 20]
3 [100]


## Topic 5: The `nonlocal` Declaration


### 1. What It Is
The `nonlocal` declaration flags a variable inside a nested function as a ***free variable***, allowing assignment (`=`, `+=`) to rebind the variable in the outer closure scope rather than creating a new local variable.

### 2. The Mechanism: Scope Lookup Order
When Python encounters a variable `x`:
1. If declared `global x`: Binds directly to the module global scope [passage 441, 454].
2. If declared `nonlocal x`: Searches enclosing function scopes top-down for `x` and binds directly to that closure cell [passage 453, 454].
3. If assigned without `nonlocal` / `global`: Compiles `x` as a local variable [passage 440, 451, 454].
4. If unassigned and referenced: Looks up `x` in enclosing scopes, then module global, then `__builtins__.__dict__` [passage 454, 455].

### 3. Gotchas & Edge Cases
* **Missing Outer Binding:** Declaring `nonlocal x` when `x` does not exist in any enclosing function scope raises a `SyntaxError: no binding for nonlocal 'x' found`. `nonlocal` cannot bind to module global variables.


In [56]:
def make_average_better():
    count, total = 0, 0 # only need to store the count and running total
    def inner(n):
        nonlocal count, total #NOTE: without nonlocal it would fail/try to find count and total from the function local scope!
        count += 1
        total += n
        return total/count
    return inner

avg_bet = make_average_better()
for i in range(10, 21):
    print(avg_bet(i))


10.0
10.5
11.0
11.5
12.0
12.5
13.0
13.5
14.0
14.5
15.0


In [57]:
### Challenge 5: Predict the Output

def make_counter(start):
    count = start
    def increment():
        nonlocal count
        count += 1
        return count
    return increment

c1 = make_counter(5)

print(1, c1()) # 6
print(2, c1()) # 7



1 6
2 7


## Topic 6: Implementing a Simple Decorator (`clock` & `functools.wraps`)


### 1. What It Is
A standard wrapper decorator captures the decorated function via closure, wraps its execution with pre/post processing, and uses `functools.wraps` to preserve the original function's identity [passage 455, 459, 460].

### 2. The Mechanism: Attribute Preservation via `@wraps`
Without `@functools.wraps(func)`, the decorated function loses its original `__name__`, `__doc__`, `__annotations__`, and parameter signature [passage 460].
`@functools.wraps(func)` is a helper decorator applied to the inner wrapper function that copies key metadata attributes from `func` to `wrapper` [passage 460, 462].

### 3. Gotchas & Edge Cases
* **Variadic Arguments (`*args, **kwargs`):** Wrappers must accept `*args, **kwargs` and pass them into `func(*args, **kwargs)` to properly support functions with arbitrary positional and keyword parameters.


In [ ]:
from functools import cache
from time import perf_counter

def naive_fib(n):
    if n < 2:
        return n
    return naive_fib(n-1) + naive_fib(n-2)

@cache
def cache_fib(n):
    if n < 2:
        return n
    return cache_fib(n-1) + cache_fib(n-2)

n = 25
t0 = perf_counter()
naive_fib(n)
print(f'naive_fib({n}) used {perf_counter() - t0}')

t1 = perf_counter()
cache_fib(n)
print(f'cache_fib({n}) used {perf_counter() - t1}')


naive_fib(25) used 0.016610916005447507
cache_fib(25) used 7.91249331086874e-05


In [58]:
### Challenge 6: Predict the Output

import functools

def my_deco(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs).upper()
    return wrapper

@my_deco
def greet(name):
    """Returns a greeting."""
    return f"hello {name}"

print(1, greet("Alice")) # HELLO ALICE
print(2, greet.__name__, greet.__doc__) # greet, Returns a greeting




1 HELLO ALICE
2 greet Returns a greeting.


## Topic 7: Standard Library Decorators: Memoization (`@cache` & `@lru_cache`)

### 1. What It Is
`functools.cache` (Python 3.9+) and `functools.lru_cache` implement memoization—caching function results based on call arguments to avoid recalculating expensive operations [passage 462, 463, 468].

### 2. The Mechanism: Dict Keys and Eviction
- **Hashable Prerequisite:** Arguments passed to a cached function are used as dictionary keys to store results [passage 467]. All arguments **must be hashable** [passage 467].
- **`lru_cache` Bounds:** `lru_cache(maxsize=128, typed=False)` discards the Least Recently Used entries when full [passage 469, 470].
- **`cache` Unboundedness:** `cache` is equivalent to `lru_cache(maxsize=None)` [passage 468, 470].

### 3. Gotchas & Edge Cases
* **Unhashable Argument Error:** Passing unhashable objects (e.g., `list`, `dict`) to a `@cache` or `@lru_cache` function raises `TypeError: unhashable type` at runtime [passage 467].
* **Memory Leaks:** Using `@cache` in long-running processes without bounding `maxsize` can consume all system memory [passage 468].


In [ ]:
### Challenge 7: Predict the Output


import functools

@functools.lru_cache(maxsize=2)
def compute(x):
    print(f"CALC {x}")
    return x * 10

#NOTE: LRU = Least RECENTLY used (not least frequently used!)
print(1, compute(1)) # 1 is added to the cache 
print(2, compute(1)) # 1 is cache hit
print(3, compute(2)) # 2 is added to cache, this makes 1 the least recently used 
print(4, compute(3)) # 3 is added to cache and 1 is evicted because it's least recently used
print(5, compute(1)) # 1 is added to cache again and 2 is evicted because it was least recently used between 2 and 3.

# CALC 1
# 1 10
# 2 10
# CALC 2
# 3 20
# CALC 3
# 4 30
# CALC 1
# 5 10


CALC 1
1 10
2 10
CALC 2
3 20
CALC 3
4 30
CALC 1
5 10


## Topic 8: Standard Library Decorators: Single Dispatch (`@singledispatch`)

### 1. What It Is
`functools.singledispatch` transforms a plain function into a generic function that dispatches call execution to specialized functions based on the type of its **first positional argument** [passage 476, 477].

### 2. The Mechanism: Registration and Inheritance Lookup
1. Base function decorated with `@singledispatch` [passage 476, 477].
2. Specialized implementations registered using `@base.register` [passage 477, 478].
3. Types can be specified via type hints on the first parameter (`def _(x: int):`) or explicitly in the decorator (`@base.register(int)`) [passage 477, 478, 480].
4. Selection algorithm finds the most specific matching class or ABC in the type hierarchy [passage 479, 481, 482].
5. The advantage of `@singledispatch` is supporting modular extension: each module can register a specialized function for each type it supports.

### 3. Gotchas & Edge Cases
* **First Argument Only:** `singledispatch` dispatches **strictly** on the type of parameter #1 [passage 476]. It does not perform multi-argument dispatch [passage 476].

### 4. Currency Check (Grounded Only)
* **BOOK vs CURRENT (Python 3.11):**
  * **BOOK (p. 327):** Describes `singledispatch` supporting type hints and explicit type arguments [passage 477, 478, 480].
  * **CURRENT (Python 3.11 Documentation):** `@functools.singledispatch` explicitly supports `types.UnionType` (`int | float`) and `typing.Union` in parameter annotations [passage 27, 28].


In [72]:
### Challenge 8: Predict the Output


from functools import singledispatch

@singledispatch
def process(arg):
    return f"default: {arg}"

@process.register(int | float)
def _(arg):
    return f"int | float: {arg * 2}"

@process.register
def _(arg: list):
    return f"list len: {len(arg)}"

print(1, process("hello")) # default: hello
print(2, process(10)) # int: 20
print(2, process(12.4)) # int | float: 24.8
print(3, process([])) # list len: 0


1 default: hello
2 int | float: 20
2 int | float: 24.8
3 list len: 0



## Topic 9: Parameterized Decorators (Decorator Factories & Class-Based)

### 1. What It Is
To make a decorator accept custom arguments, write a **decorator factory**: a function that accepts custom parameters and returns a decorator, which in turn accepts the target function and returns the wrapper [passage 484, 486].

### 2. The Mechanism: Invocation & Three-Tier Structure

### Function-Based Decorator Factory (3 Levels of Nesting):
```python
def repeat(num_times):             # Level 1: Factory (accepts custom params)
    def decorator(func):           # Level 2: Actual Decorator (accepts func)
        def wrapper(*args):        # Level 3: Wrapper (accepts call args)
            for _ in range(num_times):
                func(*args)
        return wrapper
    return decorator
```

### Class-Based Decorator Factory:
A class with `__init__` storing configuration parameters and `__call__` acting as the decorator that returns the wrapper function [passage 495].

### 3. Gotchas & Edge Cases
* **Mandatory Parentheses:** Parameterized decorators **must** be called with parentheses (`@repeat()` or `@repeat(3)`), because Python evaluates the expression after `@` first to retrieve the decorator function [passage 486, 487].




In [73]:
### Challenge 9: Predict the Output


def prefix_logger(prefix="LOG"):
    def decorator(func):
        def wrapper(*args):
            return f"[{prefix}] {func(*args)}"
        return wrapper
    return decorator

@prefix_logger("INFO")
def get_status():
    return "OK"

@prefix_logger()
def get_error():
    return "FAIL"

print(1, get_status()) # [INFO] OK
print(2, get_error()) # [LOG] FAIL


1 [INFO] OK
2 [LOG] FAIL


IMPORT: decorating action_a
IMPORT: wrapping action_b
--- START MAIN ---
RUNTIME: calling action_b
--- END MAIN ---


1 100
2 UnboundLocalError
3 300
